In [1]:
import os
import pandas as pd
import numpy as np

INPUT_PATH  = "../../../data/phase2/labeled_signals.parquet"
OUTPUT_PATH = "../../../data/phase2/features_for_model.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"
META_COLS = ["date", "s3_key", "fold", "split"]

In [2]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived and encoded columns to df. Returns a new DataFrame.
    Input df must contain: ema_20, ema_50, rsi_14, macd_hist, atr_14,
    session_quality, direction, signal_valid, label.
    """
    out = df.copy()
    out["ema_ratio"]           = out["ema_20"] / out["ema_50"]
    out["session_quality_enc"] = out["session_quality"].map({"high": 2, "medium": 1, "low": 0})
    out["direction_enc"]       = out["direction"].map({"buy": 1, "sell": -1, "none": 0})
    out["signal_valid_enc"]    = out["signal_valid"].astype(int)
    return out


def assign_walk_forward_folds(
    df: pd.DataFrame,
    train_months: int = 6,
    test_months: int = 1,
) -> pd.DataFrame:
    """
    Assign walk-forward fold metadata to each row.

    For each fold N:
      - test:  rows where date falls in [fold_start + train_months,
                                         fold_start + train_months + test_months)
      - train: rows where date falls in [fold_start, fold_start + train_months)
               AND the row is not already assigned to a test split

    Rows that don't fall into any fold's test window get fold=-1, split="unused".
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["fold"]  = -1
    df["split"] = "unused"

    min_date = df["date"].min().to_period("M")
    max_date = df["date"].max().to_period("M")

    month_period = df["date"].dt.to_period("M")

    # First pass: assign test splits (each row belongs to at most one test window)
    fold_idx = 0
    cursor = min_date
    fold_windows = []
    while True:
        train_start = cursor
        train_end   = cursor + train_months
        test_start  = train_end
        test_end    = train_end + test_months

        if test_end > max_date + 1:
            break

        fold_windows.append((fold_idx, train_start, train_end, test_start, test_end))
        test_mask = (month_period >= test_start) & (month_period < test_end)
        df.loc[test_mask, "fold"]  = fold_idx
        df.loc[test_mask, "split"] = "test"

        fold_idx += 1
        cursor += test_months

    # Second pass: assign train splits only to rows not already marked as test
    for fold_idx, train_start, train_end, test_start, test_end in fold_windows:
        train_mask = (
            (month_period >= train_start) &
            (month_period <  train_end) &
            (df["split"] != "test")
        )
        df.loc[train_mask, "fold"]  = fold_idx
        df.loc[train_mask, "split"] = "train"

    return df


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature prep pipeline:
    1. Drop rows with NaN label (direction=="none")
    2. Encode features
    3. Assign walk-forward folds
    4. Return DataFrame with META_COLS + FEATURE_COLS + LABEL_COL

    Does NOT scale features — scaling happens inside each fold during training
    to prevent leakage.
    """
    df = df[df["label"].notna()].copy()
    df = encode_features(df)
    df = assign_walk_forward_folds(df)
    keep = META_COLS + FEATURE_COLS + [LABEL_COL]
    return df[keep].reset_index(drop=True)

In [3]:
raw = pd.read_parquet(INPUT_PATH)
prepped = prepare_features(raw)

print(f"Total rows after dropping direction=none: {len(prepped)}")
print(f"\nFold distribution:")
print(prepped.groupby(["fold", "split"]).size().to_string())
print(f"\nLabel distribution:\n{prepped['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
prepped.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Total rows after dropping direction=none: 441

Fold distribution:
fold  split
0     test      51
      train     61
1     test      56
      train     57
2     test      35
      train     25
3     test      33
      train    123

Label distribution:
label
0.0    268
1.0    173
Name: count, dtype: int64

Saved to ../../../data/phase2/features_for_model.parquet


/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_69341/3995693699.py:36: UserWarning: Converting to Period representation will drop timezone information.
  min_date = df["date"].min().to_period("M")
/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_69341/3995693699.py:37: UserWarning: Converting to Period representation will drop timezone information.
  max_date = df["date"].max().to_period("M")
/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_69341/3995693699.py:39: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_period = df["date"].dt.to_period("M")


In [4]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[META_COLS + FEATURE_COLS + [LABEL_COL]].head(10).to_string())
print(f"\nFeature dtypes:\n{df[FEATURE_COLS].dtypes}")
print(f"\nAny NaN in features: {df[FEATURE_COLS].isna().any().any()}")

                       date  s3_key  fold  split  ema_ratio     rsi_14  macd_hist    atr_14  session_quality_enc  direction_enc  signal_valid_enc  label
0 2025-03-03 00:00:00+00:00  EURUSD     0  train   0.999384  48.968128  -0.000155  0.008613                    0             -1                 0    1.0
1 2025-03-05 00:00:00+00:00  EURUSD     0  train   1.000730  63.956032   0.001044  0.010542                    0              1                 0    1.0
2 2025-03-06 00:00:00+00:00  EURUSD     0  train   1.002522  71.286718   0.002672  0.011389                    0              1                 0    0.0
3 2025-03-07 00:00:00+00:00  EURUSD     0  train   1.004030  70.643024   0.003482  0.011315                    0              1                 0    1.0
4 2025-03-10 00:00:00+00:00  EURUSD     0  train   1.005708  73.338537   0.004237  0.011118                    0              1                 0    0.0
5 2025-03-11 00:00:00+00:00  EURUSD     0  train   1.007028  71.282489   0.004308 